<a href="https://colab.research.google.com/github/amit-sahu-a11y/ML_projects_for_practice/blob/main/parse.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

To install a Python library in Colab, you can use `pip install` with an exclamation mark prefix within a code cell. For example, to install the `requests` library, you would run:

In [ ]:
!pip install requests

After installation, you can import and use the library in subsequent cells:

In [ ]:
import requests

response = requests.get('https://www.google.com')
print(response.status_code)
print(response)


200
<Response [200]>


In [ ]:
!pip install beautifulsoup4

Now that `beautifulsoup4` is installed, you can import `BeautifulSoup` and use it to parse the `response.text` from your previous `requests.get` call:

In [ ]:
from bs4 import BeautifulSoup

# Assuming 'response' variable still holds the requests.get() result
soup = BeautifulSoup(response.text, 'html.parser')

# For example, let's print the title of the page
print(f"Page title: {soup.title.string}")

# Or find all links
# for link in soup.find_all('a'):
#     print(link.get('href'))

Page title: Google


Let's change our target URL to a Wikipedia page with a table to demonstrate how to scrape tabular data. We'll use the 'List of Python libraries' page as an example.

Now that we have the HTML content of a page with tables, we can find and extract data from a specific table. Wikipedia pages often have multiple tables, so we might need to identify the correct one. I'll look for the first table on the page and extract its header and row data.

It appears the cells that scraped the table data were deleted. Let's recreate them to ensure `df_table` is available.

In [ ]:
# Re-fetching the page and parsing it to create soup_tables
import requests
from bs4 import BeautifulSoup

# Trying a more relevant URL for a list of Python software.
url_with_tables = 'https://en.wikipedia.org/wiki/List_of_Python_software'
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}
response_tables = requests.get(url_with_tables, headers=headers)
response_tables.raise_for_status() # Raise an exception for bad status codes

soup_tables = BeautifulSoup(response_tables.text, 'html.parser')
print(f"Successfully fetched and parsed: {url_with_tables}")

Successfully fetched and parsed: https://en.wikipedia.org/wiki/List_of_Python_software


In [ ]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
from io import StringIO

# Correcting the URL to a valid Wikipedia page that contains comparison tables.
# Changing to a known simpler Wikipedia table to diagnose 'No tables found' issue.
url_with_tables = 'https://en.wikipedia.org/wiki/List_of_U.S._states_and_territories_by_population'

# Define User-Agent header
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}

print(f"Attempting to scrape tables directly from URL: {url_with_tables}")

df_table = pd.DataFrame() # Initialize df_table as an empty DataFrame

try:
    # Use pandas.read_html directly on the URL
    # Pass the User-Agent header via storage_options
    # Try different flavors if default doesn't work
    all_tables = pd.read_html(url_with_tables, header=0, storage_options=headers, flavor='lxml')

    if all_tables:
        print(f"Successfully found {len(all_tables)} tables using pd.read_html directly from the URL.")
        # Take the first table, which is usually the main data table on such Wikipedia pages
        df_table = all_tables[0]
        print("DataFrame 'df_table' created successfully from the first table.")
        display(df_table.head())
    else:
        print("pd.read_html found no tables on the page.")

except Exception as e:
    print(f"An error occurred while trying to read HTML tables: {e}")
    print("This might mean the URL does not contain accessible HTML tables, or there's a problem with the page content.")


# Note: The BeautifulSoup and requests logic that was previously here
# is being removed/commented out to simplify the debugging and focus
# on the pd.read_html direct approach, given the 'no <table> tag' issue.
# If this direct approach also fails, we will need to reconsider the URL
# or explore more advanced scraping techniques (e.g., headless browser).


Attempting to scrape tables directly from URL: https://en.wikipedia.org/wiki/List_of_U.S._states_and_territories_by_population
Successfully found 6 tables using pd.read_html directly from the URL.
DataFrame 'df_table' created successfully from the first table.


,Unnamed: 0,State or territory,Census population[8][9][a],Census population[8][9][a].1,House seats[b],House seats[b].1,Pop. per elec. vote (2020)[c],Pop. per seat (2020)[a],% US (2020),% EC (2020)
0,NaN,State or territory,"July 1, 2025 (est.)","April 1, 2020",Seats,%,Pop. per elec. vote (2020)[c],Pop. per seat (2020)[a],% US (2020),% EC (2020)
1,1.0,California,39355309,39538223,52,11.95%,732189,760350,11.800%,10.04%
2,2.0,Texas,31709821,29145505,38,8.74%,728638,766987,8.698%,7.43%
3,3.0,Florida,23462518,21538187,28,6.44%,717940,769221,6.428%,5.58%
4,4.0,New York,20002427,20201249,26,5.98%,721473,776971,6.029%,5.20%


In [ ]:
import pandas as pd

# Define the regex pattern to find any string containing content within square brackets
# The r'' prefix denotes a raw string, preventing issues with backslashes.
# \[\] matches a literal opening square bracket
# .* matches any character (except newline) zero or more times
# \]\[ matches a literal closing square bracket
bracket_pattern = r'\[.*\]'
# the bracket_pattern just with regex=True to idetify rows
# -------
# Apply str.contains with regex=True to identify rows matching the pattern
rows_with_brackets = df_table['State or territory'].str.contains(bracket_pattern, regex=True, na=False)

# Display the rows that contain the bracket pattern
display(df_table[rows_with_brackets])

,Unnamed: 0,State or territory,Census population[8][9][a],Census population[8][9][a].1,House seats[b],House seats[b].1,Pop. per elec. vote (2020)[c],Pop. per seat (2020)[a],% US (2020),% EC (2020)
53,53.0,Guam[11],NaN,153836,1* [note 1],—,—,—,0.046%,—
54,54.0,U.S. Virgin Islands[12],NaN,87146,1* [note 1],—,—,—,0.026%,—
55,55.0,American Samoa[13],NaN,49710,1* [note 1],—,—,—,0.015%,—
56,56.0,Northern Mariana Islands[14],NaN,47329,1* [note 1],—,—,—,0.014%,—


In [ ]:
df_states = df_table[~rows_with_brackets]


Now that you have your new DataFrame, `df_states`, how would you inspect it to confirm that it only contains state data and that all the bracketed entries have been successfully removed?

In [ ]:
# Display the first few rows of df_table to verify the data
display(df_table.head())



,Unnamed: 0,State or territory,Census population[8][9][a],Census population[8][9][a].1,House seats[b],House seats[b].1,Pop. per elec. vote (2020)[c],Pop. per seat (2020)[a],% US (2020),% EC (2020)
0,NaN,State or territory,"July 1, 2025 (est.)","April 1, 2020",Seats,%,Pop. per elec. vote (2020)[c],Pop. per seat (2020)[a],% US (2020),% EC (2020)
1,1.0,California,39355309,39538223,52,11.95%,732189,760350,11.800%,10.04%
2,2.0,Texas,31709821,29145505,38,8.74%,728638,766987,8.698%,7.43%
3,3.0,Florida,23462518,21538187,28,6.44%,717940,769221,6.428%,5.58%
4,4.0,New York,20002427,20201249,26,5.98%,721473,776971,6.029%,5.20%


In [1]:
unique_states = df_states['State or territory'].unique()
print("Unique 'State or territory' entries in df_states:")
# for state in unique_states:
    # print(f"- {state}")/
unique_states.bitwise_count()



NameError: name 'df_states' is not defined

In [ ]:
for index, state in enumerate(unique_states):
    print(f"Index {index}: {state}")


# this is best way to index and state names


Index 0: State or territory
Index 1: California
Index 2: Texas
Index 3: Florida
Index 4: New York
Index 5: Pennsylvania
Index 6: Illinois
Index 7: Ohio
Index 8: Georgia
Index 9: North Carolina
Index 10: Michigan
Index 11: New Jersey
Index 12: Virginia
Index 13: Washington
Index 14: Arizona
Index 15: Tennessee
Index 16: Massachusetts
Index 17: Indiana
Index 18: Missouri
Index 19: Maryland
Index 20: Colorado
Index 21: Wisconsin
Index 22: Minnesota
Index 23: South Carolina
Index 24: Alabama
Index 25: Louisiana
Index 26: Kentucky
Index 27: Oregon
Index 28: Oklahoma
Index 29: Connecticut
Index 30: Utah
Index 31: Nevada
Index 32: Iowa
Index 33: Puerto Rico
Index 34: Arkansas
Index 35: Kansas
Index 36: Mississippi
Index 37: New Mexico
Index 38: Idaho
Index 39: Nebraska
Index 40: West Virginia
Index 41: Hawaii
Index 42: New Hampshire
Index 43: Maine
Index 44: Montana
Index 45: Rhode Island
Index 46: Delaware
Index 47: South Dakota
Index 48: North Dakota
Index 49: Alaska
Index 50: District of C

In [ ]:
import pandas as pd

# Ensure df_table exists from previous steps, otherwise replace with your DataFrame
# For demonstration, let's assume df_table is already available and populated.
# If it's not, you'd need to re-run the scraping code or load sample data.

# Save the DataFrame to a CSV file
output_csv_filename = 'wikipedia_python_libraries_table.csv'
df_table.to_csv(output_csv_filename, index=False) # index=False prevents writing the DataFrame index as a column

print(f"Data successfully saved to {output_csv_filename}")

# Optionally, display the first few rows from the saved CSV to verify
# df_read_from_csv = pd.read_csv(output_csv_filename)
# display(df_read_from_csv.head())

Data successfully saved to wikipedia_python_libraries_table.csv


In [ ]:
import kagglehub
path = kagglehub.model_download('google/gemma/gemmaCpp/7b-pt-sfp/3')

KaggleApiHTTPError: 403 Client Error.

You don't have permission to access resource at URL: https://api.kaggle.com/v1/models.ModelApiService/ListModelInstanceVersionFiles. Please make sure you are authenticated if you are trying to access a private resource or a resource requiring consent.

In [ ]:
# This cell should be run before using kagglehub if you're getting 403 errors
from google.colab import userdata
import os

# Create a .kaggle directory if it doesn't exist
!mkdir -p ~/.kaggle

# Save Kaggle API key to a file
# It's generally safer to use secrets for sensitive info like API keys
# Assume you've set 'KAGGLE_USERNAME' and 'KAGGLE_KEY' in Colab secrets
# This creates the kaggle.json file expected by kaggle-api
with open('/root/.kaggle/kaggle.json', 'w') as f:
    f.write('{"username":"' + userdata.get('KAGGLE_USERNAME') + '","key":"' + userdata.get('KAGGLE_KEY') + '"}')

# Set permissions for the kaggle.json file
!chmod 600 ~/.kaggle/kaggle.json

print("Kaggle API credentials set up successfully.")


SecretNotFoundError: Secret KAGGLE_USERNAME does not exist.

After running the above cell to configure your Kaggle credentials, you should be able to run the `kagglehub.model_download` command without the 403 error. Try running the cell where you called `kagglehub.model_download` again.

In [ ]:
from bs4 import BeautifulSoup

# Assuming soup_tables object is available from previous steps
# If not, ensure d3ac2a3c is run to create soup_tables

# Get all elements that have a 'class' attribute
all_elements_with_class = soup_tables.find_all(class_=True)

unique_classes = set()
for element in all_elements_with_class:
    if 'class' in element.attrs:
        for cls in element['class']:
            unique_classes.add(cls)

print("Unique CSS classes found in the HTML:")
for cls in sorted(list(unique_classes)):
    print(f"- {cls}")

Unique CSS classes found in the HTML:
- Inline-Template
- Template-Fact
- Z3988
- action-view
- after-portlet
- after-portlet-lang
- ambox
- ambox-Refimprove
- ambox-content
- anonymous-show
- autocollapse
- box-More_citations_needed
- catlinks
- cdx-button
- cdx-button--action-progressive
- cdx-button--fake-button
- cdx-button--fake-button--enabled
- cdx-button--icon-only
- cdx-button--size-large
- cdx-button--weight-quiet
- cdx-button__icon
- cdx-search-input
- cdx-search-input--has-end-button
- cdx-search-input__end-button
- cdx-search-input__input-wrapper
- cdx-text-input
- cdx-text-input--has-start-icon
- cdx-text-input__icon
- cdx-text-input__input
- cdx-text-input__start-icon
- cdx-typeahead-search
- cdx-typeahead-search--auto-expand-width
- cdx-typeahead-search--show-thumbnail
- citation
- cite-bracket
- client-nojs
- cs1
- cs2
- date
- date-container
- div-col
- emptyPortlet
- external
- extiw
- firstHeading
- free
- hatnote
- hide-when-compact
- hlist
- id-lock-subscription
-